# 📊 Notebook 03: Programación Dinámica
## Algoritmos y Estructuras de Datos
**Universidad de Talca** | Facultad de Ingeniería  
**Docente:** PhD. César Astudillo  
**Semestre:** _________ | **Fecha:** _________

---
> 🎯 *Este notebook está diseñado para ser ejecutado en clase de forma interactiva.  
> Ejecuta las celdas en orden de arriba hacia abajo.*

In [ ]:
# Verificación de dependencias — ejecutar primero
# Todo el notebook usa la librería estándar de Python (visualizaciones en texto y
# pruebas con unittest); ipywidgets solo se usa en los widgets interactivos.
import sys
required = {
    'ipywidgets': 'ipywidgets',
}
for nombre, paquete in required.items():
    try:
        __import__(paquete)
        print(f"✅ {nombre} instalado correctamente")
    except ImportError:
        print(f"❌ {nombre} NO encontrado — instala con: pip install {paquete}")
print("\n🐍 Python", sys.version.split()[0], "| Todo listo para comenzar.")

## 🎯 Objetivos de Aprendizaje

Al finalizar esta sesión, el estudiante será capaz de:

1. **Comprender** la diferencia clave entre Divide y Vencerás y Programación Dinámica: los subproblemas solapados.
2. **Distinguir** y aplicar las dos estrategias de DP: top-down (memoización) y bottom-up (tabulación).
3. **Implementar** en Python las cuatro versiones de Fibonacci y comparar sus tiempos reales.
4. **Resolver** la Mochila 0/1 con DP, mostrando la tabla completa y rastreando la solución.
5. **Reconocer** la subestructura óptima en problemas de edición de texto (Levenshtein) y subsecuencias (LCS).

## 🌍 Motivación: El Problema del Cálculo Repetido

Imagina que te piden calcular el 40° número de Fibonacci a mano.  
Si usas la fórmula recursiva $F(n) = F(n-1) + F(n-2)$, empiezas a calcular… y te das cuenta de que
**calculas $F(20)$ miles de veces**. ¿No sería más inteligente *escribirlo en un papel* la primera vez
y consultarlo después?

Esa es la idea central de la **Programación Dinámica**:

> *Si un subproblema ya fue resuelto, guarda la respuesta y reutilízala en lugar de recalcularla.*

```
         fib(5)
        /      \
    fib(4)     fib(3)        ← fib(3) se calcula DOS veces
    /    \     /    \
 fib(3) fib(2) fib(2) fib(1)  ← fib(2) se calcula TRES veces
 /  \   /  \
f(2) f(1) f(1) f(0)
/  \
f(1) f(0)
```

Para $F(40)$: sin DP se hacen **~331 millones** de llamadas. Con DP: **41 sumas**.

> 📌 **Definición:** La **Programación Dinámica (DP)** es un paradigma de diseño que resuelve problemas
> de optimización mediante la combinación de soluciones a **subproblemas solapados**, almacenándolas
> en una tabla para evitar recálculo.

> 🎙️ **[PAUSA PROFESOR]** Pregunta: *"¿Cuál es la diferencia entre D&V y DP?"*  
> *(D&V: subproblemas independientes → no vale la pena guardar. DP: subproblemas solapados → guardar es esencial.)*

## 📐 Teoría: El Paradigma de Programación Dinámica

### Contraste clave con Divide y Vencerás (del PDF)

| Característica | Divide y Vencerás | Programación Dinámica |
|----------------|------------------|----------------------|
| Subproblemas | **Independientes** (no se solapan) | **Solapados** (se repiten) |
| Guardar resultados | No es necesario | **Esencial** — es la clave del paradigma |
| Ejemplo clásico | Merge Sort, findMax | Fibonacci, Mochila 0/1, LCS |
| Complejidad | $O(n \log n)$ típico | $O(n^2)$ o $O(n \cdot W)$ típico |

### Los 3 pasos del paradigma (del PDF)

1. **Plantear** la solución recursiva (identificar la relación de recurrencia)
2. **Organizar** los subproblemas en una tabla (arreglo o matriz)
3. **Consultar** la tabla antes de recurrir (si ya está → retornar; si no → calcular y guardar)

### Dos estrategias de implementación

> 📌 **Top-Down (Memoización):** Implementar la función recursiva normalmente, pero antes de calcular,
> revisar si el resultado ya está en la tabla. Si está → retornar. Si no → calcular, guardar y retornar.

> 📌 **Bottom-Up (Tabulación):** Llenar la tabla desde los subproblemas más pequeños hacia los más grandes,
> sin usar recursión. Al llegar al problema original, ya están todos los subproblemas resueltos.

| | Top-Down | Bottom-Up |
|-|----------|----------|
| Implementación | Natural (modifica recursión) | Requiere pensar el orden de llenado |
| Subproblemas calculados | Solo los necesarios | Todos (incluso innecesarios) |
| Stack de recursión | Sí (puede causar RecursionError) | No |
| Lectura del código | Más clara | Más eficiente |

> 💡 **Insight:** En Python, `functools.lru_cache` hace el top-down automáticamente con una línea de código.

### Condición necesaria: Principio de Optimalidad de Bellman

> 📌 **Principio de Bellman:** Una solución óptima a un problema contiene soluciones óptimas a
> sus subproblemas. Si esta condición no se cumple, DP **no** puede aplicarse correctamente.

> 🎙️ **[PAUSA PROFESOR]** *"¿Hay algún problema donde la solución óptima global NO contiene soluciones
> óptimas a sus subproblemas?"*  
> *(Sí: el camino más largo sin ciclos en un grafo con pesos negativos. Pero la mayoría de problemas de optimización sí cumplen Bellman.)*

In [ ]:
# ── Las 4 versiones de Fibonacci ──────────────────────────────────────────
# Versión 1: Recursivo ingenuo (del PDF) — O(2ⁿ)
# Versión 2: Top-down con lru_cache — O(n)
# Versión 3: Top-down manual — O(n)
# Versión 4: Bottom-up tabulado (del PDF) — O(n), O(1) espacio
from functools import lru_cache

# ── V1: Recursivo ingenuo — código del PDF traducido a Python ─────────────
def fib_recursivo(n: int) -> int:
    """
    Fibonacci recursivo ingenuo — traducción directa del pseudocódigo del PDF.

    Complejidad:
        Temporal: O(2ⁿ) — cada llamada genera dos llamadas; subproblemas solapados
        Espacial: O(n)   — profundidad de la pila de recursión
    """
    if n <= 1:
        return n
    return fib_recursivo(n - 1) + fib_recursivo(n - 2)


# ── V2: Top-Down con lru_cache (1 línea de DP) ────────────────────────────
@lru_cache(maxsize=None)
def fib_cache(n: int) -> int:
    """
    Fibonacci top-down con memoización automática via lru_cache.
    La función es idéntica a la recursiva; lru_cache agrega la tabla.

    Complejidad:
        Temporal: O(n) — cada subproblema se resuelve exactamente una vez
        Espacial: O(n)  — tabla de memoización
    """
    if n <= 1:
        return n
    return fib_cache(n - 1) + fib_cache(n - 2)


# ── V3: Top-Down manual con diccionario ───────────────────────────────────
def fib_memo(n: int, memo: dict = None) -> int:
    """
    Fibonacci top-down con memoización manual.
    Muestra explícitamente el patrón: consultar → calcular → guardar.

    Complejidad:
        Temporal: O(n)
        Espacial: O(n) — diccionario de memoización
    """
    if memo is None:
        memo = {}
    if n <= 1:
        return n
    if n in memo:           # Paso 3 del paradigma: consultar antes de calcular
        return memo[n]
    memo[n] = fib_memo(n - 1, memo) + fib_memo(n - 2, memo)  # calcular
    return memo[n]          # guardar ya ocurrió en la línea anterior


# ── V4: Bottom-Up tabulado — código del PDF traducido a Python ────────────
def fib_dp(n: int) -> int:
    """
    Fibonacci bottom-up con tabla iterativa — versión del PDF en Python.
    Llena la tabla desde F(0) hasta F(n), sin recursión.

    Complejidad:
        Temporal: O(n)
        Espacial: O(1) — solo guarda los dos últimos valores

    Ejemplo:
        >>> fib_dp(10)
        55
    """
    if n <= 1:
        return n
    prev2, prev1 = 0, 1   # F(0), F(1)
    for _ in range(2, n + 1):
        actual = prev1 + prev2
        prev2  = prev1
        prev1  = actual
    return prev1


# ── Verificación cruzada ──────────────────────────────────────────────────
print("Verificación: las 4 versiones dan el mismo resultado")
print(f"{'n':>4} {'Recursivo':>12} {'lru_cache':>12} {'Memo':>12} {'DP':>12}")
print("-" * 56)
for n in [0, 1, 5, 10, 15, 20]:
    v1 = fib_recursivo(n)
    v2 = fib_cache(n)
    v3 = fib_memo(n)
    v4 = fib_dp(n)
    ok = "✅" if v1 == v2 == v3 == v4 else "❌"
    print(f"{n:>4} {v1:>12} {v2:>12} {v3:>12} {v4:>12} {ok}")

In [ ]:
# Árbol de recursión de Fibonacci en texto — las llamadas repetidas se marcan con ✗
# Hace visualmente obvio por qué la versión ingenua es O(2ⁿ)
N_VIZ = 6   # fib(6) — árbol manejable para visualizar


def arbol_fib(n: int, vistos: set, contador: list, prefijo: str = '', conector: str = '') -> None:
    """
    Imprime el árbol de llamadas de fib_recursivo(n) al estilo del comando `tree`.

    La primera llamada con cada valor de n queda sin marca; las siguientes son
    trabajo repetido y se marcan con ✗.

    Parámetros:
        vistos (set):    valores de n ya calculados alguna vez
        contador (list): [llamadas totales, llamadas repetidas], se actualiza en el lugar
    """
    repetida = n in vistos
    vistos.add(n)
    contador[0] += 1
    contador[1] += repetida
    print(f"{prefijo}{conector}fib({n}){'   ✗ repetida' if repetida else ''}")
    if n > 1:
        sangria = prefijo + {'': '', '├── ': '│   ', '└── ': '    '}[conector]
        arbol_fib(n - 1, vistos, contador, sangria, '├── ')
        arbol_fib(n - 2, vistos, contador, sangria, '└── ')


contador = [0, 0]
print(f'Árbol de recursión — fib({N_VIZ}) sin memoización\n')
arbol_fib(N_VIZ, set(), contador)
total, repetidos = contador
print(f'\nTotal de llamadas: {total}  |  Repetidas: {repetidos} ({repetidos/total*100:.0f}%)')
print(f"\n💡 Para fib({N_VIZ}): {total} llamadas totales, {repetidos} son trabajo desperdiciado.")
print(f"   Con DP: solo {N_VIZ+1} operaciones. El árbol se 'aplana' en una lista lineal.")

In [ ]:
# Comparación real de tiempos — las 4 versiones para n=35
import math
import timeit

N_TEST = 35
fib_cache.cache_clear()  # limpiar caché para medir desde cero

# Medir tiempos (fib_recursivo es lento, pocas repeticiones)
t_rec  = timeit.timeit(lambda: fib_recursivo(N_TEST), number=3) / 3
t_lru  = timeit.timeit(lambda: (fib_cache.cache_clear(), fib_cache(N_TEST)), number=100) / 100
t_memo = timeit.timeit(lambda: fib_memo(N_TEST), number=100) / 100
t_dp   = timeit.timeit(lambda: fib_dp(N_TEST), number=10000) / 10000

print(f"Tiempos para fib({N_TEST}):")
print(f"  V1 Recursivo ingenuo:   {t_rec*1000:10.3f} ms")
print(f"  V2 Top-down lru_cache:  {t_lru*1000:10.5f} ms")
print(f"  V3 Top-down manual:     {t_memo*1000:10.5f} ms")
print(f"  V4 Bottom-up DP:        {t_dp*1000:10.6f} ms")
print(f"\n  Speedup V1→V4: {t_rec/t_dp:,.0f}x")

# Gráfico de barras en texto, en escala logarítmica para que se vean todas las barras:
# el largo es proporcional a log₁₀ del tiempo medido en nanosegundos
def mostrar_barras(filas, unidad="ms", ancho=40, formato="8.2f", maximo=None):
    """
    Imprime un gráfico de barras horizontal en texto.

    Parámetros:
        filas (list[tuple[str, float]]): pares (etiqueta, valor).
        unidad (str): unidad que se muestra junto a cada valor.
        ancho (int): caracteres de la barra que representa a `maximo`.
        formato (str): formato del valor impreso, por ejemplo "8.2f" o "8,.0f".
        maximo (float | None): valor que ocupa todo el ancho; por defecto, el mayor
            de `filas`. Pasar el mismo valor a dos gráficos los deja en la misma escala.
    """
    maximo = maximo or max(valor for _, valor in filas) or 1
    margen = max(len(etiqueta) for etiqueta, _ in filas)
    for etiqueta, valor in filas:
        barra = "█" * round(valor / maximo * ancho) or ("▏" if valor > 0 else "")
        print(f"{etiqueta:>{margen}} │{barra:<{ancho}} {valor:{formato}} {unidad}")


print(f'\nComparación de tiempos para fib({N_TEST}) — valor = log₁₀(tiempo en ns)\n')
mostrar_barras([('V1 Recursivo O(2ⁿ)', math.log10(t_rec * 1e9)),
                ('V2 lru_cache O(n)', math.log10(t_lru * 1e9)),
                ('V3 Memo manual O(n)', math.log10(t_memo * 1e9)),
                ('V4 Bottom-Up O(n), O(1) esp.', math.log10(t_dp * 1e9))],
               unidad='', formato='5.2f')


In [ ]:
# ── Mochila 0/1 con Programación Dinámica ────────────────────────────────
# Mismos datos que el Notebook 02 — para comparar Greedy vs DP

# Datos del PDF (idénticos a NB 02, 04, 05)
OBJETOS = [
    {'nombre': 'Objeto 1', 'valor': 20, 'peso': 50},
    {'nombre': 'Objeto 2', 'valor': 24, 'peso': 100},
    {'nombre': 'Objeto 3', 'valor': 55, 'peso': 150},
    {'nombre': 'Objeto 4', 'valor': 40, 'peso': 200},
    {'nombre': 'Objeto 5', 'valor': 70, 'peso': 250},
]
CAPACIDAD_W = 300


def mochila_01_dp(objetos: list, capacidad: int,
                  verbose: bool = False) -> dict:
    """
    Resuelve la Mochila 0/1 usando Programación Dinámica (bottom-up).
    Cada objeto se puede incluir (1) o excluir (0) — no se puede fraccionar.

    Parámetros:
        objetos (list):  lista de dicts con 'nombre', 'valor', 'peso'
        capacidad (int): peso máximo de la mochila
        verbose (bool):  si True, imprime la tabla DP completa

    Retorna:
        dict: {'valor_optimo': int, 'objetos_incluidos': list, 'tabla_dp': list}

    Complejidad:
        Temporal: O(n · W) — n filas × W columnas
        Espacial: O(n · W) — para la tabla completa (permite reconstrucción)

    Recurrencia:
        dp[i][w] = max valor con los primeros i objetos y capacidad w
        dp[i][w] = dp[i-1][w]                              si peso[i] > w
        dp[i][w] = max(dp[i-1][w], dp[i-1][w-peso[i]] + valor[i])  si no
    """
    n = len(objetos)
    # Tabla DP: (n+1) filas × (capacidad+1) columnas, inicializada en 0
    dp = [[0] * (capacidad + 1) for _ in range(n + 1)]

    # Llenar la tabla bottom-up
    for i in range(1, n + 1):
        peso_i  = objetos[i-1]['peso']
        valor_i = objetos[i-1]['valor']
        for w in range(capacidad + 1):
            # Opción 1: no incluir el objeto i
            dp[i][w] = dp[i-1][w]
            # Opción 2: incluir el objeto i (si cabe)
            if peso_i <= w:
                con_obj_i = dp[i-1][w - peso_i] + valor_i
                dp[i][w]  = max(dp[i][w], con_obj_i)

    # Reconstruir cuáles objetos se incluyen (backtracking sobre la tabla)
    incluidos = []
    w = capacidad
    for i in range(n, 0, -1):
        if dp[i][w] != dp[i-1][w]:   # el objeto i fue incluido
            incluidos.append(objetos[i-1]['nombre'])
            w -= objetos[i-1]['peso']
    incluidos.reverse()

    if verbose:
        print("Tabla DP (filas = objetos, columnas = capacidad):")
        header = "     " + " ".join(f"{c:4d}" for c in range(0, capacidad+1, 50))
        print(header)
        print("  0: " + " ".join(f"{dp[0][c]:4d}" for c in range(0, capacidad+1, 50)))
        for i in range(1, n+1):
            row = " ".join(f"{dp[i][c]:4d}" for c in range(0, capacidad+1, 50))
            print(f"  {i}: {row}   ← {objetos[i-1]['nombre']}")

    return {
        'valor_optimo':     dp[n][capacidad],
        'objetos_incluidos': incluidos,
        'tabla_dp':          dp
    }


# ── Demo ──────────────────────────────────────────────────────────────────
print("Mochila 0/1 con DP — datos del PDF (mismos que NB 02 y 04)")
print(f"Objetos: {[(o['nombre'], o['valor'], o['peso']) for o in OBJETOS]}")
print(f"Capacidad W = {CAPACIDAD_W}\n")
resultado = mochila_01_dp(OBJETOS, CAPACIDAD_W, verbose=True)
print(f"\n✅ Valor óptimo 0/1 con DP: {resultado['valor_optimo']}")
print(f"   Objetos incluidos: {resultado['objetos_incluidos']}")

# Comparar con Greedy (del NB 02)
def greedy_ratio(objetos, cap):
    """Greedy 0/1 con criterio ratio v/w (del NB 02)."""
    ords = sorted(objetos, key=lambda o: o['valor']/o['peso'], reverse=True)
    p, v, sel = cap, 0, []
    for o in ords:
        if o['peso'] <= p:
            p -= o['peso']; v += o['valor']; sel.append(o['nombre'])
    return v, sel

val_greedy, sel_greedy = greedy_ratio(OBJETOS, CAPACIDAD_W)
print(f"\n📊 Comparación:")
print(f"   Greedy 0/1 (ratio v/w): valor = {val_greedy}, selección = {sel_greedy}")
print(f"   DP óptimo:              valor = {resultado['valor_optimo']}, selección = {resultado['objetos_incluidos']}")
gap = resultado['valor_optimo'] - val_greedy
print(f"   Diferencia: {gap} ({'DP mejor' if gap > 0 else 'igual'})")

In [ ]:
# Tabla DP de la mochila en texto
# Columnas cada 50 unidades de capacidad; [x] marca las celdas del camino de reconstrucción.
resultado_viz = mochila_01_dp(OBJETOS, CAPACIDAD_W)
tabla = resultado_viz['tabla_dp']
n, W  = len(OBJETOS), CAPACIDAD_W
cols  = list(range(0, W + 1, 50))

# Camino de reconstrucción: desde dp[n][W] hacia arriba, restando el peso de cada objeto incluido
camino, w = set(), W
for i in range(n, 0, -1):
    camino.add((i, w))
    if tabla[i][w] != tabla[i-1][w]:
        w -= OBJETOS[i-1]['peso']
camino.add((0, w))

etiquetas = ['∅'] + [f"{o['nombre']} (v={o['valor']}, p={o['peso']})" for o in OBJETOS]
margen = max(len(e) for e in etiquetas)
print(f'Tabla DP — Mochila 0/1 (W={W}, columnas cada 50 unidades)\n')
print(f"{'w →':>{margen}} │" + ''.join(f'{c:>7}' for c in cols))
print('─' * (margen + 1) + '┼' + '─' * 7 * len(cols))
for i in range(n + 1):
    celdas = [f'[{tabla[i][c]}]' if (i, c) in camino else str(tabla[i][c]) for c in cols]
    print(f'{etiquetas[i]:>{margen}} │' + ''.join(f'{x:>7}' for x in celdas))

print(f'\nÓptimo: dp[{n}][{W}] = {tabla[n][W]}')
print("\n💡 Cada celda dp[i][w] = máximo valor posible usando los primeros i objetos con capacidad w.")
print("   La respuesta está en la esquina inferior derecha; los [x] muestran cómo se reconstruye.")

In [ ]:
# ── Distancia de Edición (Levenshtein) ────────────────────────────────────
# Número mínimo de operaciones (insertar, borrar, reemplazar) para transformar
# una cadena en otra. Ejemplo clásico de DP en 2D.

def levenshtein(s1: str, s2: str, verbose: bool = False) -> int:
    """
    Calcula la distancia de edición mínima entre dos cadenas (Levenshtein).
    Usa DP bottom-up con tabla 2D.

    Parámetros:
        s1 (str): cadena origen
        s2 (str): cadena destino
        verbose (bool): si True, imprime la tabla DP

    Retorna:
        int: número mínimo de operaciones (insertar, borrar, reemplazar)

    Complejidad:
        Temporal: O(m · n) donde m=len(s1), n=len(s2)
        Espacial: O(m · n) — tabla completa

    Recurrencia:
        dp[i][j] = 0                            si i=0 o j=0 (casos base)
        dp[i][j] = dp[i-1][j-1]                si s1[i-1] == s2[j-1]
        dp[i][j] = 1 + min(dp[i-1][j],         # borrar de s1
                            dp[i][j-1],          # insertar en s1
                            dp[i-1][j-1])        # reemplazar
    """
    m, n = len(s1), len(s2)
    # Crear tabla (m+1) × (n+1)
    dp = [[0] * (n + 1) for _ in range(m + 1)]

    # Casos base: transformar cadena vacía
    for i in range(m + 1): dp[i][0] = i   # borrar i caracteres
    for j in range(n + 1): dp[0][j] = j   # insertar j caracteres

    # Llenar la tabla
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if s1[i-1] == s2[j-1]:        # caracteres iguales — no cuesta nada
                dp[i][j] = dp[i-1][j-1]
            else:                          # tomar el mínimo de las 3 operaciones
                dp[i][j] = 1 + min(
                    dp[i-1][j],    # borrar s1[i]
                    dp[i][j-1],    # insertar s2[j] en s1
                    dp[i-1][j-1]   # reemplazar s1[i] por s2[j]
                )

    if verbose:
        print(f"Tabla DP para '{s1}' → '{s2}':")
        header = "    ε  " + "  ".join(s2)
        print(header)
        for i in range(m + 1):
            c = 'ε' if i == 0 else s1[i-1]
            row = "  ".join(f"{dp[i][j]:2d}" for j in range(n + 1))
            print(f"  {c}  {row}")

    return dp[m][n]


# ── Demo ──────────────────────────────────────────────────────────────────
pares = [
    ('algoritmo', 'logaritmo', 'casi iguales'),
    ('perro',     'gato',      'palabras distintas'),
    ('hola',      'hola',      'idénticas'),
    ('',          'python',    'cadena vacía a python'),
]
for s1, s2, desc in pares:
    d = levenshtein(s1, s2)
    print(f"  levenshtein('{s1}', '{s2}') = {d}  ({desc})")

print("\n🔍 Tabla detallada para 'perro' → 'gato':")
levenshtein('perro', 'gato', verbose=True)

In [ ]:
# ── Longest Common Subsequence (LCS) ─────────────────────────────────────
# Longest Common Subsequence: la subsecuencia más larga común a dos secuencias.
# Una subsecuencia NO necesita ser contigua.

def lcs(s1: str, s2: str) -> tuple:
    """
    Calcula la subsecuencia común más larga (LCS) de s1 y s2.

    Parámetros:
        s1 (str): primera cadena
        s2 (str): segunda cadena

    Retorna:
        tuple: (longitud_lcs, cadena_lcs)

    Complejidad:
        Temporal: O(m · n)
        Espacial: O(m · n)

    Recurrencia:
        dp[i][j] = 0                          si i=0 o j=0
        dp[i][j] = dp[i-1][j-1] + 1           si s1[i-1] == s2[j-1]
        dp[i][j] = max(dp[i-1][j], dp[i][j-1]) si no

    Ejemplo:
        >>> lcs('ABCBDAB', 'BDCAB')
        (4, 'BCAB')
    """
    m, n = len(s1), len(s2)
    dp   = [[0] * (n + 1) for _ in range(m + 1)]

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if s1[i-1] == s2[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
            else:
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])

    # Reconstruir la subsecuencia
    subsec = []
    i, j = m, n
    while i > 0 and j > 0:
        if s1[i-1] == s2[j-1]:
            subsec.append(s1[i-1])
            i -= 1; j -= 1
        elif dp[i-1][j] >= dp[i][j-1]:
            i -= 1
        else:
            j -= 1
    subsec.reverse()

    return dp[m][n], ''.join(subsec)


# ── Demo ──────────────────────────────────────────────────────────────────
ejemplos = [
    ('ABCBDAB',   'BDCAB',     'Ejemplo clásico CLRS'),
    ('AGGTAB',    'GXTXAYB',   'Ejemplo con ADN simulado'),
    ('algoritmo', 'ritmo',     'Una es subcadena de la otra'),
    ('AACGT',     'ACGT',      'Cadenas de ADN cortas'),
]
print(f"{'s1':>12}  {'s2':>12}  {'LCS longitud':>14}  {'LCS':>12}")
print('-' * 60)
for s1, s2, desc in ejemplos:
    lng, subsec = lcs(s1, s2)
    print(f"{s1:>12}  {s2:>12}  {lng:>14}  {subsec:>12}   ({desc})")

In [ ]:
# Traza en texto: tabla DP de Levenshtein llenándose celda a celda
S1_ANIM, S2_ANIM = 'algo', 'alga'  # palabras cortas para seguir cada paso


def traza_levenshtein(s1: str, s2: str) -> int:
    """
    Llena la tabla de Levenshtein imprimiendo la regla usada en cada celda
    y al final muestra la tabla completa.

    Retorna:
        int: distancia de edición entre s1 y s2
    """
    m, n = len(s1), len(s2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1): dp[i][0] = i
    for j in range(n + 1): dp[0][j] = j
    print(f"Levenshtein('{s1}', '{s2}') — casos base: fila 0 = 0..{n}, columna 0 = 0..{m}\n")

    paso = 0
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            paso += 1
            if s1[i-1] == s2[j-1]:
                dp[i][j] = dp[i-1][j-1]
                regla = f"'{s1[i-1]}' = '{s2[j-1]}' → copia la diagonal dp[{i-1}][{j-1}]"
            else:
                dp[i][j] = 1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])
                regla = (f"'{s1[i-1]}' ≠ '{s2[j-1]}' → 1 + min(arriba {dp[i-1][j]}, "
                         f"izquierda {dp[i][j-1]}, diagonal {dp[i-1][j-1]})")
            print(f'Paso {paso:>2}: dp[{i}][{j}] = {dp[i][j]}   {regla}')

    print('\nTabla final:\n')
    print('     ' + ' '.join(f'{c:>2}' for c in ['ε'] + list(s2)))
    for i in range(m + 1):
        print(f"  {'ε' if i == 0 else s1[i-1]}  " + ' '.join(f'{dp[i][j]:>2}' for j in range(n + 1)))
    print(f'\nDistancia de edición: {dp[m][n]}')
    return dp[m][n]


traza_levenshtein(S1_ANIM, S2_ANIM)

## 📈 Análisis de Complejidad

| Algoritmo | Temporal | Espacial | Comparado con alternativa |
|-----------|---------|---------|---------------------------|
| Fibonacci recursivo | $O(2^n)$ | $O(n)$ | — |
| Fibonacci DP (top/bottom) | $O(n)$ | $O(n)$ | $2^{35}/35 \approx 10^9\times$ mejor |
| Fibonacci DP $O(1)$ esp. | $O(n)$ | $O(1)$ | Óptimo en espacio |
| Mochila 0/1 DP | $O(n \cdot W)$ | $O(n \cdot W)$ | vs $O(2^n)$ fuerza bruta |
| Levenshtein DP | $O(m \cdot n)$ | $O(m \cdot n)$ | vs $O(3^{mn})$ sin DP |
| LCS DP | $O(m \cdot n)$ | $O(m \cdot n)$ | vs $O(2^m \cdot n)$ sin DP |

> ⚠️ **Importante:** La Mochila 0/1 con DP es **pseudo-polinomial**: $O(n \cdot W)$ depende del valor de $W$,
> no solo del número de objetos. Si $W$ es muy grande (ej. $10^9$), la tabla no cabe en memoria.
> En ese caso se usan técnicas de compresión o aproximación.

> 💡 **Insight:** La DP transforma problemas exponenciales en polinomiales al *reutilizar* subproblemas.
> El precio es espacio adicional: $O(n)$ para Fibonacci, $O(n \cdot W)$ para la Mochila.

In [ ]:
# Widget interactivo: comparador de llamadas Recursivo vs DP para Fibonacci
# Auto-contenido — no necesita celdas anteriores.
import ipywidgets as widgets
from IPython.display import display
import math

def _fib_rec_contar(n):
    """Devuelve (valor, llamadas) para Fibonacci recursivo ingenuo."""
    cont = [0]
    def f(k):
        cont[0] += 1
        if k <= 1: return k
        return f(k-1) + f(k-2)
    val = f(n)
    return val, cont[0]

slider_n = widgets.IntSlider(value=10, min=2, max=35, step=1,
                              description='n:', continuous_update=False,
                              style={'description_width':'initial'},
                              layout=widgets.Layout(width='400px'))
boton  = widgets.Button(description='▶ Comparar', button_style='primary')
salida = widgets.Output()

def al_comparar(b):
    with salida:
        salida.clear_output(wait=True)
        n = slider_n.value

        # Contar llamadas recursivas
        if n <= 30:
            _, llamadas_rec = _fib_rec_contar(n)
        else:
            llamadas_rec = 2**(n+1)  # aproximación para n grandes
            print(f"  (n={n}>30: estimando llamadas recursivas como 2^n≈{llamadas_rec:,})")
        llamadas_dp = n + 1  # DP calcula exactamente n+1 subproblemas

        print(f"fib({n}):")
        print(f"  Recursivo:  {llamadas_rec:>15,} llamadas")
        print(f"  DP:         {llamadas_dp:>15,} operaciones")
        print(f"  Ratio:      {llamadas_rec/llamadas_dp:>15,.1f}x más trabajo")

        # Crecimiento hasta n en texto (escala logarítmica: largo ∝ log₁₀ de las llamadas)
        print(f"\n{'k':>4} │ {'llamadas recursivas (escala log)':<40} {'recursivo':>12} {'DP':>4}")
        for k in range(2, n + 1):
            llamadas = _fib_rec_contar(k)[1] if k <= 25 else 2 ** k
            barra = '█' * round(math.log10(llamadas) / math.log10(max(llamadas_rec, 10)) * 40)
            print(f'{k:>4} │ {barra:<40} {llamadas:>12,} {k + 1:>4}')

boton.on_click(al_comparar)
display(widgets.VBox([
    widgets.HTML('<h4 style="color:#212121">📊 Comparador Recursivo vs DP — Fibonacci</h4>'),
    slider_n, boton, salida
]))

## 🧪 Ejercicio 1: Fibonacci Bottom-Up con O(1) Espacio ⭐

**Descripción:** Implementa la función de Fibonacci usando DP bottom-up con **O(1) de espacio**.
En lugar de guardar toda la tabla, guarda únicamente los dos últimos valores calculados.

Luego responde experimentalmente: **¿para qué valor de n la versión recursiva ingenua tarda
más de 5 segundos en tu computador?**

**Entrada:** entero `n` (0 ≤ n ≤ 1000 para la versión DP)

**Salida:** $F(n)$ como entero (Python maneja enteros de precisión arbitraria)

**Ejemplo:**
```
Entrada: n = 10
Salida:  55
```

**Complejidad esperada:** $O(n)$ temporal, $O(1)$ espacial

In [ ]:
def fibonacci_opt(n: int) -> int:
    """
    Calcula el n-ésimo número de Fibonacci usando DP bottom-up con O(1) espacio.
    Solo guarda los dos últimos valores calculados.

    Parámetros:
        n (int): índice de Fibonacci (n >= 0)

    Retorna:
        int: F(n) (entero de precisión arbitraria)

    Complejidad:
        Temporal: O(n)
        Espacial: O(1) — solo 2 variables
    """
    # Tu código aquí
    # Pista: usa dos variables prev2=F(i-2) y prev1=F(i-1), actualiza en cada paso.
    pass

In [ ]:
# Pruebas con unittest
import timeit
import unittest


class TestFibonacciOpt(unittest.TestCase):
    """fibonacci_opt(n) calcula F(n) con DP bottom-up."""

    CASOS = [
        (0, 0, 'F(0) = 0'),
        (1, 1, 'F(1) = 1'),
        (2, 1, 'F(2) = 1'),
        (10, 55, 'F(10) = 55'),
        (20, 6765, 'F(20) = 6765'),
        (50, 12586269025, 'F(50) — entero grande'),
    ]

    def test_casos(self):
        """valores conocidos de Fibonacci"""
        for n, esperado, descripcion in self.CASOS:
            with self.subTest(descripcion):
                self.assertEqual(fibonacci_opt(n), esperado)

    def test_n_1000(self):
        """F(1000) tiene 209 dígitos y se calcula al instante"""
        self.assertEqual(len(str(fibonacci_opt(1000))), 209)


resultado = unittest.main(argv=['ignorado', 'TestFibonacciOpt'], exit=False, verbosity=2)

if resultado.result.wasSuccessful():
    # Bonus: medir cuándo la versión recursiva se vuelve intolerable
    def _rec(k):
        if k <= 1: return k
        return _rec(k-1) + _rec(k-2)
    print()
    for k in [30, 33, 35, 38]:
        t = timeit.timeit(lambda kk=k: _rec(kk), number=1)
        print(f"  Recursivo fib({k:2d}): {t:.3f}s")

In [ ]:
# ═══════════════════════════════════════════════════
# SOLUCIÓN — Descomenta para ver después de intentarlo
# ═══════════════════════════════════════════════════

# def fibonacci_opt(n):
#     """Fibonacci DP bottom-up con O(1) espacio."""
#     if n <= 1:
#         return n
#     prev2, prev1 = 0, 1   # F(0), F(1)
#     for _ in range(2, n + 1):
#         actual = prev1 + prev2  # F(i) = F(i-1) + F(i-2)
#         prev2  = prev1          # avanzar ventana
#         prev1  = actual
#     return prev1
#
# # Complejidad: O(n) temporal, O(1) espacial.
# # Python soporta enteros de precisión arbitraria:
# # fibonacci_opt(1000) calcula un número de 209 dígitos sin problema.

## 🧪 Ejercicio 2: Mochila 0/1 con DP ⭐⭐

**Descripción:** Resuelve el problema de la Mochila 0/1 con Programación Dinámica.
Usa los **mismos 5 objetos del Notebook 02** para poder comparar directamente la solución
Greedy (subóptima para 0/1) con la solución DP (óptima).

**Entrada:** lista de tuplas `(peso, valor)` + capacidad `W`

**Salida:** tupla `(valor_máximo, lista_de_índices_incluidos)`

**Ejemplo:**
```
Entrada: pesos=[50,100,150,200,250], valores=[20,24,55,40,70], W=300
Salida:  (99, [0, 1, 2])   # Objetos 1, 2 y 3: peso 50+100+150 = 300, valor 20+24+55 = 99
```

**Complejidad esperada:** $O(n \cdot W)$ temporal y espacial

In [ ]:
def mochila_01(pesos: list, valores: list, W: int) -> tuple:
    """
    Resuelve la Mochila 0/1 con DP bottom-up.

    Parámetros:
        pesos (list):   lista de pesos de los objetos
        valores (list): lista de valores de los objetos
        W (int):        capacidad máxima de la mochila

    Retorna:
        tuple: (valor_maximo, lista_de_indices_0based)

    Complejidad:
        Temporal: O(n · W)
        Espacial: O(n · W)
    """
    # Tu código aquí
    # Pista: construye tabla dp[i][w] = máximo valor con primeros i objetos y capacidad w
    # Luego reconstruye qué objetos se incluyeron haciendo backtracking sobre la tabla.
    pass

In [ ]:
# Pruebas con unittest
class TestMochila01(unittest.TestCase):
    """mochila_01(pesos, valores, W) retorna (valor_máximo, índices incluidos)."""

    CASOS = [
        ([50, 100, 150, 200, 250], [20, 24, 55, 40, 70], 300, 99, 'datos del PDF (W=300)'),
        ([10, 20, 30], [60, 100, 120], 50, 220, 'ejemplo clásico CLRS (W=50)'),
        ([5], [10], 5, 10, 'un solo objeto que cabe'),
        ([10], [10], 5, 0, 'un solo objeto que no cabe'),
        ([1, 2, 3], [6, 10, 12], 5, 22, 'tres objetos, W=5'),
    ]

    def test_valor_optimo(self):
        """el valor retornado es el óptimo"""
        for pesos, valores, W, esperado, descripcion in self.CASOS:
            with self.subTest(descripcion):
                valor, _ = mochila_01(pesos, valores, W)
                self.assertEqual(valor, esperado)

    def test_indices_coherentes(self):
        """los índices son distintos, caben en la mochila y suman el valor informado"""
        for pesos, valores, W, _, descripcion in self.CASOS:
            with self.subTest(descripcion):
                valor, indices = mochila_01(pesos, valores, W)
                self.assertEqual(len(set(indices)), len(indices))
                self.assertLessEqual(sum(pesos[i] for i in indices), W)
                self.assertEqual(sum(valores[i] for i in indices), valor)


resultado = unittest.main(argv=['ignorado', 'TestMochila01'], exit=False, verbosity=2)

In [ ]:
# ═══════════════════════════════════════════════════
# SOLUCIÓN — Descomenta para ver después de intentarlo
# ═══════════════════════════════════════════════════

# def mochila_01(pesos, valores, W):
#     """Mochila 0/1 con DP bottom-up — O(n·W)."""
#     n  = len(pesos)
#     dp = [[0]*(W+1) for _ in range(n+1)]
#
#     for i in range(1, n+1):
#         for w in range(W+1):
#             dp[i][w] = dp[i-1][w]  # no incluir objeto i
#             if pesos[i-1] <= w:    # incluir objeto i si cabe
#                 con = dp[i-1][w - pesos[i-1]] + valores[i-1]
#                 dp[i][w] = max(dp[i][w], con)
#
#     # Backtracking para reconstruir qué objetos se incluyeron
#     incluidos = []
#     w = W
#     for i in range(n, 0, -1):
#         if dp[i][w] != dp[i-1][w]:
#             incluidos.append(i-1)  # índice 0-based
#             w -= pesos[i-1]
#     incluidos.reverse()
#     return dp[n][W], incluidos

## 🧪 Ejercicio 3: Longest Increasing Subsequence (LIS) ⭐⭐⭐

**Descripción:** Dada una lista de enteros, encuentra la **subsecuencia creciente más larga** (LIS).
Una subsecuencia es creciente si cada elemento es estrictamente mayor que el anterior.
La subsecuencia no necesita ser contigua.

Implementa **dos versiones**:
- Versión $O(n^2)$: DP estándar
- Versión $O(n \log n)$: DP + búsqueda binaria (patience sorting)

**Entrada:** lista de enteros

**Salida:** tupla `(longitud_LIS, subsecuencia_LIS)`

**Ejemplo:**
```
Entrada: [10, 9, 2, 5, 3, 7, 101, 18]
Salida:  (4, [2, 5, 7, 101])  # o [2, 3, 7, 101] — hay varias LIS de longitud 4
```

**Complejidad esperada:** $O(n^2)$ la versión básica; $O(n \log n)$ la versión avanzada  
**Nivel CF equivalente:** ~1300

In [ ]:
def lis_cuadratico(arr: list) -> tuple:
    """
    Encuentra la Subsecuencia Creciente Más Larga (LIS) — versión O(n²).

    Parámetros:
        arr (list): lista de enteros

    Retorna:
        tuple: (longitud_LIS, lista_con_una_LIS)

    Complejidad:
        Temporal: O(n²)
        Espacial: O(n)

    Ejemplo:
        >>> lis_cuadratico([10, 9, 2, 5, 3, 7, 101, 18])
        (4, [2, 5, 7, 101])
    """
    # Tu código aquí
    # Pista: dp[i] = longitud de la LIS que TERMINA en arr[i]
    # Para reconstruir: guarda también 'padre[i]' = índice del elemento anterior en la LIS
    pass


def lis_nlogn(arr: list) -> int:
    """
    Encuentra la LONGITUD de la LIS — versión O(n log n).
    Usa búsqueda binaria (patience sorting) — no reconstruye la subsecuencia.

    Complejidad:
        Temporal: O(n log n)
        Espacial: O(n)
    """
    # Tu código aquí
    # Pista: mantén una lista 'tails' donde tails[i] = menor elemento final
    # de todas las subsecuencias crecientes de longitud i+1.
    # Usa bisect_left para encontrar el punto de inserción en O(log n).
    pass

In [ ]:
# Pruebas con unittest
def es_subsecuencia(sub: list, arr: list) -> bool:
    """True si los elementos de sub aparecen en arr en el mismo orden (no necesariamente contiguos)."""
    resto = iter(arr)
    return all(x in resto for x in sub)


class TestLIS(unittest.TestCase):
    """lis_cuadratico(arr) → (longitud, una LIS); lis_nlogn(arr) → longitud."""

    CASOS = [
        ([10, 9, 2, 5, 3, 7, 101, 18], 4, 'ejemplo del enunciado'),
        ([0, 1, 0, 3, 2, 3], 4, 'LIS = [0, 1, 2, 3]'),
        ([7, 7, 7, 7, 7], 1, 'todos iguales — estrictamente creciente'),
        ([1, 2, 3, 4, 5], 5, 'completamente creciente'),
        ([5, 4, 3, 2, 1], 1, 'completamente decreciente'),
        ([3, 5, 6, 2, 5, 4, 19, 5, 6, 7, 12], 6, 'caso mixto'),
    ]

    def test_cuadratico_longitud(self):
        """lis_cuadratico: la longitud es la de la LIS"""
        for arr, esperado, descripcion in self.CASOS:
            with self.subTest(descripcion):
                self.assertEqual(lis_cuadratico(arr)[0], esperado)

    def test_cuadratico_subsecuencia(self):
        """lis_cuadratico: la subsecuencia es creciente, está en arr y tiene esa longitud"""
        for arr, _, descripcion in self.CASOS:
            with self.subTest(descripcion):
                longitud, sub = lis_cuadratico(arr)
                self.assertEqual(len(sub), longitud)
                self.assertTrue(all(a < b for a, b in zip(sub, sub[1:])), f'{sub} no es creciente')
                self.assertTrue(es_subsecuencia(sub, arr), f'{sub} no es subsecuencia de {arr}')

    def test_nlogn_longitud(self):
        """lis_nlogn: la longitud es la de la LIS"""
        for arr, esperado, descripcion in self.CASOS:
            with self.subTest(descripcion):
                self.assertEqual(lis_nlogn(arr), esperado)


resultado = unittest.main(argv=['ignorado', 'TestLIS'], exit=False, verbosity=2)

In [ ]:
# ═══════════════════════════════════════════════════
# SOLUCIÓN — Descomenta para ver después de intentarlo
# ═══════════════════════════════════════════════════

# def lis_cuadratico(arr):
#     """LIS con DP O(n²) — dp[i] = longitud de LIS que termina en arr[i]."""
#     if not arr: return 0, []
#     n    = len(arr)
#     dp   = [1] * n          # toda posición es LIS de longitud 1 consigo misma
#     padre = [-1] * n        # para reconstruir el camino
#     for i in range(1, n):
#         for j in range(i):
#             if arr[j] < arr[i] and dp[j] + 1 > dp[i]:
#                 dp[i]    = dp[j] + 1
#                 padre[i] = j
#     # Encontrar el índice de la LIS más larga
#     idx_max = max(range(n), key=lambda i: dp[i])
#     # Reconstruir
#     lis = []
#     idx = idx_max
#     while idx != -1:
#         lis.append(arr[idx]); idx = padre[idx]
#     lis.reverse()
#     return dp[idx_max], lis
#
#
# def lis_nlogn(arr):
#     """Longitud de LIS con patience sorting — O(n log n)."""
#     from bisect import bisect_left
#     if not arr: return 0
#     tails = []  # tails[i] = menor fin posible de LIS de longitud i+1
#     for x in arr:
#         pos = bisect_left(tails, x)   # primera posición donde x podría reemplazar
#         if pos == len(tails):
#             tails.append(x)           # extiende la LIS más larga
#         else:
#             tails[pos] = x            # reemplaza para mantener valores menores
#     return len(tails)

## 🔬 Zona de Experimentación

Las siguientes celdas son tuyas para experimentar. Algunas sugerencias:
- ¿Cuál es el número de Fibonacci más grande que Python puede calcular en menos de 1 segundo con `fibonacci_opt`?
- ¿Puedes adaptar `mochila_01_dp` para usar $O(W)$ de espacio en vez de $O(n \cdot W)$?
- Prueba `levenshtein` con palabras en español con tildes. ¿Cómo afectan los caracteres acentuados?

In [ ]:
# Espacio libre para experimentar
# Sugerencia: adapta mochila_01_dp para usar O(W) espacio (solo 2 filas)

In [ ]:
# Espacio libre para experimentar
# Sugerencia: usa lcs() para comparar dos secuencias de ADN de 20 caracteres

In [ ]:
# Autoevaluación — 4 preguntas sobre Programación Dinámica
import ipywidgets as widgets
from IPython.display import display

preguntas = [
    {
        'pregunta': '1. ¿Cuál es la diferencia PRINCIPAL entre Divide y Vencerás y Programación Dinámica?',
        'opciones': [
            'D&V usa recursión; DP no puede usar recursión',
            'En D&V los subproblemas son independientes; en DP se solapan y se guardan en tabla',
            'DP siempre es más lento que D&V',
            'D&V solo funciona con arreglos; DP funciona con cualquier estructura'
        ],
        'correcta': 1,
        'explicacion': 'La clave es el SOLAPAMIENTO. En D&V, cada subproblema se resuelve una sola vez (son independientes). En DP, los mismos subproblemas aparecen repetidamente, por eso guardamos sus soluciones.'
    },
    {
        'pregunta': '2. ¿Qué es la "memoización" en el contexto de Programación Dinámica?',
        'opciones': [
            'Una técnica de memorización para aprender algoritmos',
            'Llenar la tabla DP de abajo hacia arriba sin recursión',
            'Guardar el resultado de una subproblema en una tabla antes de retornarlo',
            'Una estrategia de poda para reducir el árbol de búsqueda'
        ],
        'correcta': 2,
        'explicacion': 'Memoización (no "memorización") = consultar la tabla antes de calcular + guardar el resultado si no estaba. Es la implementación top-down de DP: se parece al código recursivo original pero evita recálculos.'
    },
    {
        'pregunta': '3. ¿Por qué la Mochila 0/1 con DP tiene complejidad O(n·W) y es "pseudo-polinomial"?',
        'opciones': [
            'Porque usa n filas y W columnas en la tabla, y W puede ser exponencial en los bits de entrada',
            'Porque es polinomial en n pero no en W',
            'Porque siempre tarda más que O(n²)',
            'Porque W siempre es mayor que n'
        ],
        'correcta': 0,
        'explicacion': 'O(n·W) parece polinomial, pero W es el VALOR de la capacidad, no su representación en bits. Si W=10⁹, la tabla tiene 10⁹ columnas — no cabe en memoria. Por eso se llama pseudo-polinomial: polinomial en el valor, exponencial en el tamaño de la entrada.'
    },
    {
        'pregunta': '4. ¿Qué ventaja tiene bottom-up (tabulación) sobre top-down (memoización) en Python?',
        'opciones': [
            'Bottom-up siempre usa menos memoria',
            'Bottom-up evita el límite de recursión de Python (RecursionError para n grande)',
            'Top-down nunca puede implementarse en Python',
            'Bottom-up siempre es más rápido que top-down'
        ],
        'correcta': 1,
        'explicacion': 'Python tiene un límite de profundidad de recursión (~1000 por defecto). Top-down con memoización puede alcanzarlo para n grande. Bottom-up usa un bucle iterativo y no tiene este problema. En Python, sys.setrecursionlimit() puede aumentarlo, pero bottom-up es la solución limpia.'
    }
]

def crear_quiz(preguntas):
    for i, p in enumerate(preguntas):
        radio  = widgets.RadioButtons(options=p['opciones'], description=f'P{i+1}:',
                                       style={'description_width':'initial'},
                                       layout={'width':'max-content'})
        boton  = widgets.Button(description='Verificar', button_style='info')
        salida = widgets.Output()
        label  = widgets.HTML(f'<b>{p["pregunta"]}</b>')
        def verificar(b, r=radio, out=salida,
                      c=p['correcta'], ex=p['explicacion'], opts=p['opciones']):
            with out:
                out.clear_output()
                if r.value == opts[c]:
                    print(f'✅ ¡Correcto! {ex}')
                else:
                    print(f'❌ No exactamente. Respuesta: "{opts[c]}"\n   {ex}')
        boton.on_click(verificar)
        display(widgets.VBox([label, radio, boton, salida]))
        print('─' * 65)

crear_quiz(preguntas)

## 📚 Lecturas Recomendadas y Práctica

### Textbooks

| Libro | Edición | Capítulo | Tema |
|-------|---------|----------|------|
| Cormen et al. (CLRS) | 4ª ed. | **Cap. 14** | Dynamic Programming — Rod cutting (14.1), LCS (14.4) |
| Kleinberg & Tardos | 1ª ed. | **Cap. 6** | Dynamic Programming — Weighted Interval Scheduling, Mochila |
| Skiena | 3ª ed. | **Cap. 10** | Dynamic Programming — Edit distance, TSP aproximado |

### Recursos Gratuitos

- 🌐 [VisuAlgo — DP](https://visualgo.net/en/dp) — visualización de Coin Change y LCS
- 🏆 [AtCoder Educational DP Contest](https://atcoder.jp/contests/dp) — 26 problemas graduados de DP, soporta Python 3
- 📖 [CP-Algorithms — DP](https://cp-algorithms.com/dynamic_programming/intro-to-dp.html) — explicaciones con código

### Práctica en Codeforces (soporta Python 3)

> 🔍 **Cómo filtrar:** [codeforces.com/problemset](https://codeforces.com/problemset) → Tag: `dp`

| # | Criterio de búsqueda | Rating | Por qué es útil |
|---|---------------------|--------|----------------|
| 1 | Tag `dp` + Rating 800 | ⭐ 800 | Fibonacci-like o DP 1D simple |
| 2 | Tag `dp` + Rating 1000 | ⭐⭐ 1000 | Problemas de camino mínimo o subsecuencias |
| 3 | Tag `dp` + Rating 1200 | ⭐⭐⭐ 1200 | DP 2D o combinaciones de subproblemas |
| 4 | [AtCoder DP Contest](https://atcoder.jp/contests/dp) Problemas A–E | ⭐–⭐⭐⭐ | Graduados perfectamente: Frog, Knapsack, LCS |

> ⚠️ Se recomienda **fuertemente** el AtCoder DP Contest para este tópico — es la mejor colección estructurada de problemas DP disponible en línea.

---

**Próximo notebook:** [04_backtracking.ipynb](04_backtracking.ipynb) — cuando DP tampoco alcanza, búsqueda exhaustiva con retroceso.